# Live Demo — Panel Regression
### Session 6 · ExInt II · WU Vienna · SS 2026

**Research design:**
- **Y:** RoA = `ib / at`
- **X:** R&D intensity = `xrd / at`
- **H1:** R&D intensity positively affects RoA
- **H2:** Firm size positively moderates the R&D–RoA relationship
- **Interaction:** `rd_intensity × ln_at`

**Models:**
1. Pooled OLS — baseline
2. Two-Way FE — main specification
3. TWFE + H2 interaction
4. Random Effects + Hausman comparison

> Reference: https://github.com/vkiefner/sme-intl

---
## Cell 1 — Setup & load

In [4]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS, RandomEffects
warnings.filterwarnings('ignore')

def find_env():
    current = Path(os.getcwd())
    for path in [current] + list(current.parents):
        if (path / '.env').exists(): return path / '.env'
        try:
            for s in path.iterdir():
                if s.is_dir() and (s / '.env').exists(): return s / '.env'
        except PermissionError: continue
    raise FileNotFoundError('Could not find .env')

project_root = find_env().parent
os.chdir(project_root)
print(f'Working directory: {os.getcwd()}')

df = pd.read_parquet('data/processed/panel_with_vars.parquet')
print(f'Loaded: {df.shape[0]:,} rows | {df["gvkey"].nunique():,} firms')
print(f'Years: {df["fyear"].min()} - {df["fyear"].max()}')
print(f'R&D firms: {(df["rd_intensity"]>0).sum():,} ({(df["rd_intensity"]>0).mean()*100:.1f}%)')

Working directory: /Users/thomasmacbookair/Documents/medtech-internationalization-drivers
Loaded: 1,134 rows | 166 firms
Years: 2015 - 2025
R&D firms: 855 (75.4%)


---
## Cell 2 — Define variables & build interaction term

In [5]:
# Variable names — ADAPT these for your own project
DV       = 'roa'
X_MAIN   = 'rd_intensity'
MODERATOR= 'ln_at'
CONTROLS = ['ln_at', 'leverage', 'capx_intensity', 'cash_ratio']

# H2 interaction: R&D intensity × firm size
df['rd_x_size'] = df['rd_intensity'] * df['ln_at']
INTERACT = 'rd_x_size'

# Drop rows missing any regression variable
reg_vars = [DV, X_MAIN, INTERACT] + CONTROLS
df_reg   = df.dropna(subset=reg_vars).copy()
print(f'Regression sample: {len(df_reg):,} obs | {df_reg["gvkey"].nunique():,} firms')

# Set panel index — REQUIRED by linearmodels
df_panel = df_reg.set_index(['gvkey', 'fyear'])
print('Panel index set: (gvkey, fyear)')

Regression sample: 1,134 obs | 166 firms
Panel index set: (gvkey, fyear)


---
## Cell 3 — Model 1: Pooled OLS (baseline)

Ignores panel structure. Shows the naive cross-sectional relationship.
Expected: R&D intensity negatively associated with RoA (R&D is expensed
immediately, reducing current-period earnings — especially for SMEs).

In [6]:
formula_ols = f"{DV} ~ {X_MAIN} + {' + '.join(CONTROLS)}"
print(f'Formula: {formula_ols}\n')

res1 = smf.ols(formula_ols, data=df_reg).fit(cov_type='HC3')
print(res1.summary().tables[1])
print(f'\nR² = {res1.rsquared:.3f} | N = {int(res1.nobs):,}')

Formula: roa ~ rd_intensity + ln_at + leverage + capx_intensity + cash_ratio

                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -0.5267      0.042    -12.672      0.000      -0.608      -0.445
rd_intensity      -1.1386      0.116     -9.787      0.000      -1.367      -0.911
ln_at              0.0965      0.008     11.675      0.000       0.080       0.113
leverage          -0.0322      0.094     -0.344      0.731      -0.215       0.151
capx_intensity     0.1051      0.230      0.458      0.647      -0.345       0.555
cash_ratio        -0.1360      0.045     -3.035      0.002      -0.224      -0.048

R² = 0.319 | N = 1,134


---
## Cell 4 — Model 2: Two-Way Fixed Effects (main specification)

Firm FE absorbs time-invariant firm characteristics (management quality,
industry, country). Year FE absorbs common macro shocks (COVID, financial
crisis). Clustered SEs correct for within-firm error correlation over time.

**This is the model you report as your main result.**

In [7]:
formula_fe = (f"{DV} ~ {X_MAIN} + {' + '.join(CONTROLS)} "
              f"+ EntityEffects + TimeEffects")
print(f'Formula: {formula_fe}\n')

res2 = PanelOLS.from_formula(formula_fe, data=df_panel).fit(
    cov_type='clustered', cluster_entity=True
)
print(res2.summary.tables[1])
print(f'\nR² (within) = {res2.rsquared:.3f} | N = {int(res2.nobs):,}')

Formula: roa ~ rd_intensity + ln_at + leverage + capx_intensity + cash_ratio + EntityEffects + TimeEffects

                               Parameter Estimates                                
                Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------------------------------------------------------------------------------
rd_intensity      -0.8969     0.2253    -3.9804     0.0001     -1.3391     -0.4547
ln_at              0.2196     0.0441     4.9756     0.0000      0.1330      0.3063
leverage          -0.2300     0.1713    -1.3428     0.1797     -0.5662      0.1061
capx_intensity    -0.0911     0.2601    -0.3503     0.7262     -0.6016      0.4194
cash_ratio         0.0949     0.0897     1.0579     0.2904     -0.0811      0.2708

R² (within) = 0.296 | N = 1,134


---
## Cell 5 — Model 3: TWFE + Interaction (H2)

Does firm size moderate the R&D–RoA relationship?

**H2:** β(rd_intensity × ln_at) > 0  
If positive: larger SMEs benefit more from R&D investment than smaller ones
(they can spread fixed R&D costs over a larger asset base and have better
absorptive capacity to commercialise R&D outputs).

In [8]:
formula_int = (f"{DV} ~ {X_MAIN} + {INTERACT} + {' + '.join(CONTROLS)} "
               f"+ EntityEffects + TimeEffects")
print(f'Formula: {formula_int}\n')

res3 = PanelOLS.from_formula(formula_int, data=df_panel).fit(
    cov_type='clustered', cluster_entity=True
)
print(res3.summary.tables[1])
print(f'\nR² (within) = {res3.rsquared:.3f} | N = {int(res3.nobs):,}')

Formula: roa ~ rd_intensity + rd_x_size + ln_at + leverage + capx_intensity + cash_ratio + EntityEffects + TimeEffects

                               Parameter Estimates                                
                Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------------------------------------------------------------------------------
rd_intensity      -1.5542     0.3777    -4.1155     0.0000     -2.2954     -0.8131
rd_x_size          0.2822     0.1475     1.9132     0.0560     -0.0073      0.5717
ln_at              0.1940     0.0399     4.8643     0.0000      0.1157      0.2723
leverage          -0.2475     0.1733    -1.4285     0.1535     -0.5875      0.0925
capx_intensity    -0.1078     0.2527    -0.4267     0.6697     -0.6038      0.3881
cash_ratio         0.1110     0.0913     1.2158     0.2243     -0.0682      0.2903

R² (within) = 0.311 | N = 1,134


---
## Cell 6 — Random Effects + Hausman comparison

**Logic:** If unobserved firm-level heterogeneity is uncorrelated with
R&D intensity, RE is more efficient than FE. If correlated, FE is
consistent and RE is biased.

**In IB research:** we almost always prefer FE because firm-level
unobservables (management quality, culture) very plausibly correlate
with R&D investment decisions.

In [10]:
formula_re = f"{DV} ~ {X_MAIN} + {' + '.join(CONTROLS)}"
res_re = RandomEffects.from_formula(formula_re, data=df_panel).fit()

fe_coef = res2.params[X_MAIN]
re_coef = res_re.params[X_MAIN]
diff    = abs(fe_coef - re_coef)

print(f'Hausman comparison — coefficient on {X_MAIN}:')
print(f'  FE:   {fe_coef:.4f} (p={res2.pvalues[X_MAIN]:.3f})')
print(f'  RE:   {re_coef:.4f} (p={res_re.pvalues[X_MAIN]:.3f})')
print(f'  Diff: {diff:.4f}')
if diff > 0.005:
    print('  → Non-trivial difference → prefer FE (consistent)')
else:
    print('  → Small difference → RE may be efficient, but FE still preferred in IB')

Hausman comparison — coefficient on rd_intensity:
  FE:   -0.8969 (p=0.000)
  RE:   -1.4749 (p=0.000)
  Diff: 0.5780
  → Non-trivial difference → prefer FE (consistent)


---
## Cell 7 — Side-by-side results table

In [11]:
# ── Cell 7 — Side-by-side results table ──────────────────────────────────────

def get_se(res, var):
    """Standard errors — statsmodels uses bse, linearmodels uses std_errors."""
    if hasattr(res, 'std_errors'):
        return res.std_errors[var]   # linearmodels (PanelOLS, RandomEffects)
    return res.bse[var]              # statsmodels (OLS)

def stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

all_vars = [X_MAIN, INTERACT] + CONTROLS
models   = [res1, res2, res3]
labels   = ['(1) OLS', '(2) TWFE', '(3) TWFE+H2']

rows = []
for var in all_vars:
    row = {'Variable': var}
    for label, res in zip(labels, models):
        if var in res.params.index:
            b  = res.params[var]
            se = get_se(res, var)
            p  = res.pvalues[var]
            row[label] = f'{b:.4f}{stars(p)}\n({se:.4f})'
        else:
            row[label] = ''
    rows.append(row)

results = pd.DataFrame(rows).set_index('Variable')
results.loc['Firm FE']      = ['No',  'Yes', 'Yes']
results.loc['Year FE']      = ['No',  'Yes', 'Yes']
results.loc['Clustered SE'] = ['No',  'Yes', 'Yes']
results.loc['N']  = [f'{int(res1.nobs):,}',
                     f'{int(res2.nobs):,}',
                     f'{int(res3.nobs):,}']
results.loc['R²'] = [f'{res1.rsquared:.3f}',
                     f'{res2.rsquared:.3f}',
                     f'{res3.rsquared:.3f}']

print(results[labels].to_string())
print('\n* p<0.10  ** p<0.05  *** p<0.01')
print('SEs in parentheses. Models (2)-(3): clustered at firm level.')

                             (1) OLS              (2) TWFE           (3) TWFE+H2
Variable                                                                        
rd_intensity    -1.1386***\n(0.1163)  -0.8969***\n(0.2253)  -1.5542***\n(0.3777)
rd_x_size                                                      0.2822*\n(0.1475)
ln_at            0.0965***\n(0.0083)   0.2196***\n(0.0441)   0.1940***\n(0.0399)
leverage           -0.0322\n(0.0935)     -0.2300\n(0.1713)     -0.2475\n(0.1733)
capx_intensity      0.1051\n(0.2296)     -0.0911\n(0.2601)     -0.1078\n(0.2527)
cash_ratio      -0.1360***\n(0.0448)      0.0949\n(0.0897)      0.1110\n(0.0913)
Firm FE                           No                   Yes                   Yes
Year FE                           No                   Yes                   Yes
Clustered SE                      No                   Yes                   Yes
N                              1,134                 1,134                 1,134
R²                          

---
## Cell 8 — Interpret results

In [12]:
# ── Cell 8 — Interpret results ────────────────────────────────────────────────

b_x   = res2.params[X_MAIN]
p_x   = res2.pvalues[X_MAIN]
b_int = res3.params.get(INTERACT, np.nan)
p_int = res3.pvalues.get(INTERACT, 1)

print('=== H1: R&D intensity -> RoA (Model 2, TWFE) ===')
print(f'  β(rd_intensity) = {b_x:.4f}{stars(p_x)}  (p = {p_x:.3f})')
if p_x < 0.10:
    if b_x < 0:
        print('  H1 SUPPORTED: negative effect as predicted')
        print('  R&D expensing reduces current-period RoA under IFRS —')
        print('  costs are recognised immediately; returns accrue with a 2-5 year lag')
    else:
        print('  H1 NOT SUPPORTED: effect is positive (unexpected direction)')
        print('  Check: is R&D intensity winsorized? Any sample selection issues?')
else:
    print(f'  H1 NOT SUPPORTED: effect not significant (p={p_x:.3f})')
    print('  Consider: sample may have too few R&D firms for within-firm identification')

print(f'\n=== H2: Firm size moderates R&D -> RoA (Model 3, TWFE+H2) ===')
print(f'  β(rd_x_size) = {b_int:.4f}{stars(p_int)}  (p = {p_int:.3f})')
if p_int < 0.10:
    if b_int > 0:
        print('  H2 SUPPORTED: larger SMEs benefit more from R&D investment')
        print('  Consistent with absorptive capacity — larger firms can')
        print('  spread fixed R&D costs and commercialise outputs more efficiently')
    else:
        print('  H2 NOT SUPPORTED in expected direction: negative moderation')
        print('  Smaller firms may benefit more — or larger firms face higher R&D costs')
else:
    print(f'  H2 NOT SUPPORTED: interaction not significant (p={p_int:.3f})')

print(f'\n=== OLS vs TWFE comparison ===')
ols_b = res1.params[X_MAIN]
fe_b  = res2.params[X_MAIN]
pct_diff = abs((ols_b - fe_b) / ols_b) * 100
print(f'  OLS β = {ols_b:.4f}  |  FE β = {fe_b:.4f}  |  difference = {pct_diff:.1f}%')
print(f'  R² OLS = {res1.rsquared:.3f}  |  R² within FE = {res2.rsquared:.3f}')
if pct_diff > 20:
    print('  Large OLS-FE difference → substantial omitted variable bias in OLS')
    print('  Firm fixed effects absorb time-invariant heterogeneity')
    print('  (e.g. management quality, innovation culture) that correlates with R&D')

=== H1: R&D intensity -> RoA (Model 2, TWFE) ===
  β(rd_intensity) = -0.8969***  (p = 0.000)
  H1 SUPPORTED: negative effect as predicted
  R&D expensing reduces current-period RoA under IFRS —
  costs are recognised immediately; returns accrue with a 2-5 year lag

=== H2: Firm size moderates R&D -> RoA (Model 3, TWFE+H2) ===
  β(rd_x_size) = 0.2822*  (p = 0.056)
  H2 SUPPORTED: larger SMEs benefit more from R&D investment
  Consistent with absorptive capacity — larger firms can
  spread fixed R&D costs and commercialise outputs more efficiently

=== OLS vs TWFE comparison ===
  OLS β = -1.1386  |  FE β = -0.8969  |  difference = 21.2%
  R² OLS = 0.319  |  R² within FE = 0.296
  Large OLS-FE difference → substantial omitted variable bias in OLS
  Firm fixed effects absorb time-invariant heterogeneity
  (e.g. management quality, innovation culture) that correlates with R&D


---
## Cell 9 — Save results

In [31]:
from pathlib import Path
Path('output/tables').mkdir(parents=True, exist_ok=True)
results.to_csv('output/tables/regression_results.csv')
print('Saved: output/tables/regression_results.csv')
print('\nNext steps:')
print('  1. Adapt 03_descriptives.py and 04_regression.py for your own variables')
print('  2. git add code/ && git commit -m "session 6: descriptives and regression"')
print('  3. git push')

Saved: output/tables/regression_results.csv

Next steps:
  1. Adapt 03_descriptives.py and 04_regression.py for your own variables
  2. git add code/ && git commit -m "session 6: descriptives and regression"
  3. git push


---
## Summary — what to tell your reader

| Model | Key result | What it shows |
|-------|-----------|---------------|
| (1) OLS | β(R&D) | Naive cross-section — biased |
| (2) TWFE | β(R&D) | Within-firm effect — main result |
| (3) TWFE+H2 | β(R&D×size) | Does size moderate the R&D effect? |
| RE vs FE | Hausman diff | Justifies choice of FE |

**Reading the R² jump from OLS to FE:**  
If R²(within FE) > R²(OLS), firm fixed effects explain substantial variance
— confirming that unobserved firm heterogeneity is important and OLS is biased.

*ExInt II · WU Vienna · SS 2026 · github.com/vkiefner/sme-intl*